# VPIN Analysis on Kalshi Prediction Markets

First application of Volume-Synchronized Probability of Informed Trading (VPIN) to prediction market data.

Uses the [Becker (2026)](https://jbecker.dev/research/prediction-market-microstructure) Kalshi dataset with exact `taker_side` trade classification.

In [ ]:
# Setup: install deps and download data (~10 min)
!pip install -q duckdb pandas matplotlib scipy scikit-learn numpy pyarrow tqdm
!apt-get install -qq zstd > /dev/null 2>&1

import os
if not os.path.exists('data'):
    print('Downloading dataset (~33GB)...')
    !curl -L -o data.tar.zst https://s3.jbecker.dev/data.tar.zst
    print('Extracting...')
    !zstd -d data.tar.zst --stdout | tar -xf -
    !rm data.tar.zst
    print('Done.')
else:
    print('Data already exists.')

In [ ]:
# Quick sanity check
import duckdb
con = duckdb.connect()

n_trades = con.execute("SELECT COUNT(*) FROM 'data/kalshi/trades/*.parquet'").fetchone()[0]
n_markets = con.execute("SELECT COUNT(*) FROM 'data/kalshi/markets/*.parquet'").fetchone()[0]
print(f'Trades: {n_trades:,}')
print(f'Markets: {n_markets:,}')

# Show a sample trade
con.execute("SELECT * FROM 'data/kalshi/trades/*.parquet' LIMIT 3").df()

## VPIN Computation

Volume-bucketed order imbalance, computed entirely in DuckDB.

In [ ]:
MIN_TRADES = 500

def vpin_cte(trades_table, bucket_size=200, lookback=10):
    return f"""volume_running AS (
        SELECT *,
            SUM(count) OVER (
                PARTITION BY ticker ORDER BY created_time
            ) AS cum_vol
        FROM {trades_table}
    ),
    volume_buckets AS (
        SELECT *,
            FLOOR((cum_vol - 1) / {bucket_size}) AS bucket_id
        FROM volume_running
    ),
    bucket_stats AS (
        SELECT
            ticker,
            bucket_id,
            SUM(CASE WHEN taker_side = 'yes' THEN count ELSE 0 END) AS v_yes,
            SUM(CASE WHEN taker_side = 'no' THEN count ELSE 0 END) AS v_no,
            SUM(count) AS total_vol,
            AVG(yes_price) AS avg_price,
            MIN(created_time) AS bucket_start,
            MAX(created_time) AS bucket_end
        FROM volume_buckets
        GROUP BY ticker, bucket_id
    ),
    vpin_series AS (
        SELECT
            ticker,
            bucket_id,
            v_yes,
            v_no,
            total_vol,
            avg_price,
            bucket_start,
            bucket_end,
            ABS(v_yes - v_no)::DOUBLE / total_vol AS order_imbalance,
            AVG(ABS(v_yes - v_no)::DOUBLE / total_vol) OVER w AS vpin,
            AVG((v_yes - v_no)::DOUBLE / total_vol) OVER w AS signed_flow,
            COUNT(*) OVER w AS window_size
        FROM bucket_stats
        WINDOW w AS (
            PARTITION BY ticker ORDER BY bucket_id
            ROWS BETWEEN {lookback - 1} PRECEDING AND CURRENT ROW
        )
    )"""

In [ ]:
# Compute VPIN for all qualifying markets
BUCKET_SIZE = 200
LOOKBACK = 10

vpin_df = con.execute(f"""
    WITH market_info AS (
        SELECT ticker, result, event_ticker, close_time
        FROM 'data/kalshi/markets/*.parquet'
        WHERE status = 'finalized' AND result IN ('yes', 'no')
    ),
    qualified_markets AS (
        SELECT m.ticker
        FROM 'data/kalshi/trades/*.parquet' t
        INNER JOIN market_info m ON t.ticker = m.ticker
        GROUP BY m.ticker
        HAVING SUM(t.count) >= {MIN_TRADES}
    ),
    trades AS (
        SELECT t.ticker, t.count, t.taker_side, t.yes_price, t.created_time
        FROM 'data/kalshi/trades/*.parquet' t
        INNER JOIN qualified_markets q ON t.ticker = q.ticker
    ),
    {vpin_cte('trades', BUCKET_SIZE, LOOKBACK)}
    SELECT
        vs.*,
        m.result,
        m.event_ticker,
        m.close_time
    FROM vpin_series vs
    INNER JOIN market_info m ON vs.ticker = m.ticker
    WHERE vs.window_size = {LOOKBACK}
""").df()

print(f'Qualifying markets: {vpin_df["ticker"].nunique():,}')
print(f'Total buckets: {len(vpin_df):,}')
print(f'VPIN range: [{vpin_df["vpin"].min():.3f}, {vpin_df["vpin"].max():.3f}]')
print(f'Mean VPIN: {vpin_df["vpin"].mean():.4f}')
vpin_df.head()

## Test 1: Does VPIN Predict Volatility?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

# Compute future absolute price change at k=1,5,10
for k in [1, 5, 10]:
    vpin_df[f'future_price_{k}'] = vpin_df.groupby('ticker')['avg_price'].shift(-k)
    vpin_df[f'abs_change_{k}'] = (vpin_df[f'future_price_{k}'] - vpin_df['avg_price']).abs()

print('=== VPIN -> Future Volatility ===')
for k in [1, 5, 10]:
    valid = vpin_df[['vpin', f'abs_change_{k}']].dropna()
    slope, intercept, r, p, se = stats.linregress(valid['vpin'], valid[f'abs_change_{k}'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'  k={k:2d}: beta={slope:+.4f}{sig}  R2={r**2:.4f}  p={p:.2e}  n={len(valid):,}')

In [ ]:
# Quintile analysis
valid_k5 = vpin_df[['vpin', 'abs_change_5']].dropna()
valid_k5['quintile'] = pd.qcut(valid_k5['vpin'], 5, labels=False, duplicates='drop') + 1
quintiles = valid_k5.groupby('quintile').agg(
    mean_vpin=('vpin', 'mean'),
    mean_volatility=('abs_change_5', 'mean'),
    n=('vpin', 'count'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter
ax = axes[0]
sample = valid_k5.sample(min(5000, len(valid_k5)), random_state=42)
ax.scatter(sample['vpin'], sample['abs_change_5'], alpha=0.1, s=3, color='#3498db')
z = np.polyfit(valid_k5['vpin'], valid_k5['abs_change_5'], 1)
x_line = np.linspace(valid_k5['vpin'].min(), valid_k5['vpin'].max(), 100)
ax.plot(x_line, z[0] * x_line + z[1], color='#e74c3c', linewidth=2)
ax.set_xlabel('VPIN'); ax.set_ylabel('|Price Change| (cents, k=5)')
ax.set_title('VPIN vs Future Volatility')
ax.grid(True, alpha=0.3)

# Quintile bars
ax = axes[1]
ax.bar(quintiles['quintile'], quintiles['mean_volatility'], color='#2ecc71', alpha=0.8)
ax.set_xlabel('VPIN Quintile'); ax.set_ylabel('Mean |Price Change| (cents)')
ax.set_title('Volatility by VPIN Quintile (k=5)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
quintiles

## Test 2: Does Signed Flow Predict Direction?

In [ ]:
for k in [1, 5, 10]:
    vpin_df[f'future_return_{k}'] = vpin_df.groupby('ticker')['avg_price'].shift(-k) - vpin_df['avg_price']

print('=== Signed Flow -> Future Direction ===')
for k in [1, 5, 10]:
    valid = vpin_df[['signed_flow', f'future_return_{k}']].dropna()
    slope, intercept, r, p, se = stats.linregress(valid['signed_flow'], valid[f'future_return_{k}'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'  k={k:2d}: beta={slope:+.4f}{sig}  R2={r**2:.4f}  p={p:.2e}  n={len(valid):,}')

# Plot
valid_dir = vpin_df[['signed_flow', 'future_return_5']].dropna()
sample = valid_dir.sample(min(5000, len(valid_dir)), random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(sample['signed_flow'], sample['future_return_5'], alpha=0.1, s=3, color='#9b59b6')
z = np.polyfit(valid_dir['signed_flow'], valid_dir['future_return_5'], 1)
x_line = np.linspace(valid_dir['signed_flow'].min(), valid_dir['signed_flow'].max(), 100)
plt.plot(x_line, z[0] * x_line + z[1], color='#e74c3c', linewidth=2)
plt.xlabel('Signed Flow'); plt.ylabel('Future Return (cents, k=5)')
plt.title('Signed Flow vs Future Price Direction')
plt.grid(True, alpha=0.3)
plt.show()

## Test 3: VPIN by Category

In [ ]:
# Category mapping (simplified)
CATEGORY_PATTERNS = [
    ('NFL', 'Sports'), ('NBA', 'Sports'), ('MLB', 'Sports'), ('NHL', 'Sports'),
    ('UFC', 'Sports'), ('EPL', 'Sports'), ('F1', 'Sports'), ('PGA', 'Sports'),
    ('NCAA', 'Sports'), ('ATP', 'Sports'), ('WTA', 'Sports'), ('BOXING', 'Sports'),
    ('MLS', 'Sports'), ('NASCAR', 'Sports'), ('WNBA', 'Sports'), ('SB', 'Sports'),
    ('PRES', 'Politics'), ('SENATE', 'Politics'), ('HOUSE', 'Politics'),
    ('CABINET', 'Politics'), ('TRUMP', 'Politics'), ('BIDEN', 'Politics'),
    ('GOV', 'Politics'), ('VOTE', 'Politics'), ('ELECTION', 'Politics'),
    ('BTC', 'Crypto'), ('ETH', 'Crypto'), ('DOGE', 'Crypto'), ('SOL', 'Crypto'),
    ('XRP', 'Crypto'), ('COIN', 'Crypto'), ('SHIBA', 'Crypto'),
    ('INX', 'Finance'), ('FED', 'Finance'), ('CPI', 'Finance'), ('GDP', 'Finance'),
    ('NASDAQ', 'Finance'), ('RATE', 'Finance'), ('TNOTE', 'Finance'),
    ('PAYROLLS', 'Finance'), ('TARIFF', 'Finance'), ('EUR', 'Finance'), ('USD', 'Finance'),
    ('HIGH', 'Weather'), ('RAIN', 'Weather'), ('SNOW', 'Weather'), ('TORNADO', 'Weather'),
    ('SPOTIFY', 'Entertainment'), ('OSCAR', 'Entertainment'), ('GRAM', 'Entertainment'),
    ('RT', 'Entertainment'), ('NETFLIX', 'Entertainment'), ('BILLBOARD', 'Entertainment'),
    ('LLM', 'Science/Tech'), ('AI', 'Science/Tech'), ('SPACEX', 'Science/Tech'),
    ('POPE', 'World Events'), ('NOBEL', 'World Events'), ('EPSTEIN', 'World Events'),
]

def get_group(event_ticker):
    if not event_ticker:
        return 'Other'
    upper = event_ticker.upper()
    for pattern, group in CATEGORY_PATTERNS:
        if pattern in upper:
            return group
    return 'Other'

vpin_df['group'] = vpin_df['event_ticker'].apply(get_group)

# Category stats
cat_stats = []
for group in vpin_df['group'].unique():
    gdata = vpin_df[vpin_df['group'] == group]
    n_mkts = gdata['ticker'].nunique()
    if n_mkts < 5:
        continue
    cat_stats.append({
        'group': group,
        'mean_vpin': gdata['vpin'].mean(),
        'n_markets': n_mkts,
        'n_buckets': len(gdata),
    })

cat_df = pd.DataFrame(cat_stats).sort_values('n_buckets', ascending=False)

GROUP_COLORS = {
    'Sports': '#1f77b4', 'Politics': '#d62728', 'Crypto': '#ff7f0e',
    'Finance': '#2ca02c', 'Science/Tech': '#9467bd', 'Weather': '#17becf',
    'Entertainment': '#e377c2', 'World Events': '#8c564b', 'Other': '#aaaaaa',
}

colors = [GROUP_COLORS.get(g, '#aaaaaa') for g in cat_df['group']]
plt.figure(figsize=(10, 5))
bars = plt.bar(range(len(cat_df)), cat_df['mean_vpin'], color=colors, alpha=0.8)
plt.xticks(range(len(cat_df)), cat_df['group'], rotation=45, ha='right')
for i, row in enumerate(cat_df.itertuples()):
    plt.text(i, row.mean_vpin + 0.005, f'n={row.n_markets}', ha='center', fontsize=9)
plt.ylabel('Mean VPIN')
plt.title('Order Flow Toxicity by Category')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
cat_df

## Mispricing Predictor

Three models compared on temporal out-of-sample test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Extract midpoint features per market
market_stats = vpin_df.groupby('ticker').agg(
    max_bucket=('bucket_id', 'max'),
).reset_index()

midpoints = []
for _, row in market_stats.iterrows():
    ticker = row['ticker']
    mid = row['max_bucket'] / 2.0
    mdata = vpin_df[vpin_df['ticker'] == ticker]
    closest = mdata.iloc[(mdata['bucket_id'] - mid).abs().argsort()[:1]]
    midpoints.append(closest.iloc[0])

mid_df = pd.DataFrame(midpoints)
mid_df['y'] = (mid_df['result'] == 'yes').astype(int)
mid_df['price'] = mid_df['avg_price']
mid_df['log_volume'] = np.log1p(mid_df.groupby('ticker')['total_vol'].transform('sum'))
mid_df['close_time'] = pd.to_datetime(mid_df['close_time'])
mid_df['bucket_end'] = pd.to_datetime(mid_df['bucket_end'])
mid_df['days_to_close'] = ((mid_df['close_time'] - mid_df['bucket_end']).dt.total_seconds() / 86400).clip(lower=0)

# Temporal split
mid_df = mid_df.sort_values('close_time').reset_index(drop=True)
split = int(len(mid_df) * 0.7)
train, test = mid_df.iloc[:split].copy(), mid_df.iloc[split:].copy()

# Longshot adjustment from training set
train['price_bin'] = pd.cut(train['price'], bins=20, labels=False)
bin_stats = train.groupby('price_bin').agg(win_rate=('y', 'mean'), mean_price=('price', 'mean')).reset_index()
bin_stats['longshot_adj'] = bin_stats['win_rate'] - bin_stats['mean_price'] / 100
adj_map = dict(zip(bin_stats['price_bin'], bin_stats['longshot_adj']))
train['longshot_adj'] = train['price_bin'].map(adj_map).fillna(0)
test['price_bin'] = pd.cut(test['price'], bins=20, labels=False)
test['longshot_adj'] = test['price_bin'].map(adj_map).fillna(0)

test_y = test['y'].values
print(f'Train: {len(train)} markets, Test: {len(test)} markets')

# Model 0: Market price
p0 = np.clip(test['price'].values / 100.0, 0.01, 0.99)
brier_0 = ((p0 - test_y) ** 2).mean()

# Model 1: Logistic on price
sc1 = StandardScaler()
m1 = LogisticRegression(max_iter=1000)
m1.fit(sc1.fit_transform(train[['price']]), train['y'])
p1 = np.clip(m1.predict_proba(sc1.transform(test[['price']]))[:, 1], 0.01, 0.99)
brier_1 = ((p1 - test_y) ** 2).mean()

# Model 2: Full
feat_cols = ['price', 'vpin', 'signed_flow', 'days_to_close', 'log_volume', 'longshot_adj']
# Add category dummies
for g in mid_df['group'].unique():
    col = f'cat_{g}'
    train[col] = (train['group'] == g).astype(int)
    test[col] = (test['group'] == g).astype(int)
    feat_cols.append(col)

X_train = np.nan_to_num(train[feat_cols].values.astype(float))
X_test = np.nan_to_num(test[feat_cols].values.astype(float))
sc2 = StandardScaler()
m2 = LogisticRegression(max_iter=1000)
m2.fit(sc2.fit_transform(X_train), train['y'])
p2 = np.clip(m2.predict_proba(sc2.transform(X_test))[:, 1], 0.01, 0.99)
brier_2 = ((p2 - test_y) ** 2).mean()

# Diebold-Mariano
d = (p0 - test_y)**2 - (p2 - test_y)**2
dm_stat = d.mean() / (d.std() / np.sqrt(len(d))) if d.std() > 0 else 0
dm_p = 2 * (1 - stats.norm.cdf(abs(dm_stat)))

print(f'\n=== Brier Scores (lower = better) ===')
print(f'  Market Price:       {brier_0:.4f}')
print(f'  Longshot Correction:{brier_1:.4f}')
print(f'  Full (VPIN):        {brier_2:.4f}')
print(f'  DM stat: {dm_stat:.2f}, p={dm_p:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Brier comparison
ax = axes[0]
names = ['Market Price', 'Longshot\nCorrection', 'Full\n(VPIN)']
briers = [brier_0, brier_1, brier_2]
colors = ['#95a5a6', '#3498db', '#e74c3c']
bars = ax.bar(names, briers, color=colors, alpha=0.8)
for bar, val in zip(bars, briers):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.4f}', ha='center')
ax.set_ylabel('Brier Score')
ax.set_title('Model Comparison')
ax.grid(True, alpha=0.3, axis='y')

# Calibration
ax = axes[1]
for probs, name, color in [(p0, 'Market', '#95a5a6'), (p1, 'Longshot', '#3498db'), (p2, 'VPIN', '#e74c3c')]:
    bins = np.linspace(0, 1, 11)
    bin_idx = np.clip(np.digitize(probs, bins) - 1, 0, 9)
    cal_x, cal_y = [], []
    for b in range(10):
        mask = bin_idx == b
        if mask.sum() > 0:
            cal_x.append(probs[mask].mean())
            cal_y.append(test_y[mask].mean())
    ax.plot(cal_x, cal_y, 'o-', label=name, color=color, markersize=5)
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('Predicted'); ax.set_ylabel('Observed')
ax.set_title('Calibration Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Feature importance
ax = axes[2]
core = ['price', 'vpin', 'signed_flow', 'days_to_close', 'log_volume', 'longshot_adj']
coefs = dict(zip(feat_cols, m2.coef_[0] / sc2.scale_))
core_vals = [coefs[f] for f in core]
bar_colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in core_vals]
ax.barh(core, core_vals, color=bar_colors, alpha=0.8)
ax.set_xlabel('Coefficient')
ax.set_title('Feature Importance')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle(f'Mispricing Predictor (DM={dm_stat:.2f}, p={dm_p:.3f})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()